In [0]:
spark

In [0]:
%pip install --upgrade --force-reinstall langgraph langgraph-prebuilt langchain langchain-core databricks-langchain

Looking in indexes: [REDACTED]
  Using cached http://package-proxy.host.local:8184/pypi/v1/packages/langgraph/1.2.20/langgraph-1.2.20-py3-none-any.whl.metadata (4.9 kB)
  Using cached http://package-proxy.host.local:8184/pypi/v1/packages/langgraph-prebuilt/1.1.1/langgraph_prebuilt-1.1.1-py3-none-any.whl.metadata (6.9 kB)
  Using cached http://package-proxy.host.local:8184/pypi/v1/packages/langchain/1.4.9/langchain-1.4.9-py3-none-any.whl.metadata (6.2 kB)
  Using cached http://package-proxy.host.local:8184/pypi/v1/packages/langchain-core/1.7.6/langchain_core-1.7.6-py3-none-any.whl.metadata (4.8 kB)
  Using cached http://package-proxy.host.local:8184/pypi/v1/packages/databricks-langchain/0.22.0/databricks_langchain-0.22.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached http://package-proxy.host.local:8184/pypi/v1/packages/langgraph-checkpoint/4.3.1/langgraph_checkpoint-4.3.1-py3-none-any.whl.metadata (6.7 kB)
  Using cached http://package-proxy.host.local:8184/pypi/v1/packages/langgrap

In [0]:
dbutils.library.restartPython()

In [0]:
import importlib.metadata as m
for p in ["langgraph","langgraph-prebuilt","langchain","langchain-core","databricks-langchain"]:
    print(p, m.version(p))

langgraph 1.2.20
langgraph-prebuilt 1.1.1
langchain 1.4.9
langchain-core 1.7.6
databricks-langchain 0.22.0


In [0]:
from typing import Annotated, TypedDict
from databricks_langchain import ChatDatabricks
from langchain_core.messages import AnyMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

import os
dbutils.widgets.text("key", "")                        # leave the default empty
os.environ["KEY_HERE"] = dbutils.widgets.get("key")

from langchain.chat_models import init_chat_model
llm = init_chat_model("google_genai:gemini-2.5-flash").bind_tools(tools)

@tool
def get_weather(city: str) -> str:
    """Get tomorrow's weather forecast for a city."""
    return f"{city}: 90% chance of rain tomorrow."

@tool
def move_meeting_online(meeting: str) -> str:
    """Move a calendar meeting to an online video call."""
    return f"Done. '{meeting}' is now an online meeting."

tools = [get_weather, move_meeting_online]

# llm = ChatDatabricks(endpoint="aidemo").bind_tools(tools)

SYSTEM = SystemMessage("You are a helpful assistant. Think step by step. "
                       "Use tools when you need facts or need to do something. "
                       "Never make up tool results.")

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def think(state: State):
    return {"messages": [llm.invoke([SYSTEM] + state["messages"])]}

def should_act(state: State):
    return "act" if state["messages"][-1].tool_calls else END

graph = StateGraph(State)
graph.add_node("think", think)
graph.add_node("act", ToolNode(tools))
graph.add_edge(START, "think")
graph.add_conditional_edges("think", should_act, ["act", END])
graph.add_edge("act", "think")
agent = graph.compile()


In [0]:
dbutils.widgets.text("key", "")

In [0]:
import os
from typing import Annotated, TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# API key from the widget (never hard-code it)
key = dbutils.widgets.get("key").strip()
print("key length:", len(key))          # must be more than 0
os.environ["AQ.Ab8RN6IlSOzcJc3AWUuHZyJgc8OiDJdVr6WIpKyUUsdvltjK4A"] = key

# 1. Tools (defined BEFORE the model uses them)
@tool
def get_weather(city: str) -> str:
    """Get tomorrow's weather forecast for a city."""
    return f"{city}: 90% chance of rain tomorrow."

@tool
def move_meeting_online(meeting: str) -> str:
    """Move a calendar meeting to an online video call."""
    return f"Done. '{meeting}' is now an online meeting."

tools = [get_weather, move_meeting_online]

# 2. Model
llm = init_chat_model("google_genai:gemini-2.5-flash").bind_tools(tools)

# 3. System prompt
SYSTEM = SystemMessage("You are a helpful assistant. Think step by step. "
                       "Use tools when you need facts or need to do something. "
                       "Never make up tool results.")

# 4. State (memory)
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def think(state: State):
    return {"messages": [llm.invoke([SYSTEM] + state["messages"])]}

def should_act(state: State):
    return "act" if state["messages"][-1].tool_calls else END

graph = StateGraph(State)
graph.add_node("think", think)
graph.add_node("act", ToolNode(tools))
graph.add_edge(START, "think")
graph.add_conditional_edges("think", should_act, ["act", END])
graph.add_edge("act", "think")
agent = graph.compile()


key length: 53


In [0]:
import os
w = dbutils.widgets.get("key")
print("widget length:", len(w))
print("env length:", len(os.environ.get("AQ.Ab8RN6IlSOzcJc3AWUuHZyJgc8OiDJdVr6WIpKyUUsdvltjK4A", "")))

widget length: 53
env length: 53


In [0]:
import os, getpass
os.environ["AQ.Ab8RN6IlSOzcJc3AWUuHZyJgc8OiDJdVr6WIpKyUUsdvltjK4A"] = getpass.getpass("Gemini API key: ")
print("env length:", len(os.environ["AQ.Ab8RN6IlSOzcJc3AWUuHZyJgc8OiDJdVr6WIpKyUUsdvltjK4A"]))

Gemini API key:  [REDACTED]

env length: 53


In [0]:
import os
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6IlSOzcJc3AWUuHZyJgc8OiDJdVr6WIpKyUUsdvltjK4A"
print("env length:", len(os.environ["GOOGLE_API_KEY"]))



env length: 53


In [0]:
import os
from typing import Annotated, TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# API key from the widget (never hard-code it)
key = dbutils.widgets.get("key").strip()
if not key:
    key = os.environ.get("GOOGLE_API_KEY", "").strip()
print("key length:", len(key))
os.environ["GOOGLE_API_KEY"] = key

# 1. Tools (defined BEFORE the model uses them)
@tool
def get_weather(city: str) -> str:
    """Get tomorrow's weather forecast for a city."""
    return f"{city}: 90% chance of rain tomorrow."

@tool
def move_meeting_online(meeting: str) -> str:
    """Move a calendar meeting to an online video call."""
    return f"Done. '{meeting}' is now an online meeting."

tools = [get_weather, move_meeting_online]

# 2. Model
llm = init_chat_model("google_genai:gemini-3.8-flash").bind_tools(tools)

# 3. System prompt
SYSTEM = SystemMessage("You are a helpful assistant. Think step by step. "
                       "Use tools when you need facts or need to do something. "
                       "Never make up tool results.")

# 4. State (memory)
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def think(state: State):
    return {"messages": [llm.invoke([SYSTEM] + state["messages"])]}

def should_act(state: State):
    return "act" if state["messages"][-1].tool_calls else END

graph = StateGraph(State)
graph.add_node("think", think)
graph.add_node("act", ToolNode(tools))
graph.add_edge(START, "think")
graph.add_conditional_edges("think", should_act, ["act", END])
graph.add_edge("act", "think")
agent = graph.compile()

key length: 53


In [0]:
result = agent.invoke(
    {"messages": [("user", "Will it rain in Chennai tomorrow? If yes, move my 10am standup online.")]},
    {"recursion_limit": 10},
)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Will it rain in Chennai tomorrow? If yes, move my 10am standup online.
================================== Ai Message ==================================

[]
Tool Calls:
  get_weather (call_11057683832082357807)
 Call ID: call_11057683832082357807
  Args:
    city: Chennai
================================= Tool Message =================================
Name: get_weather

Chennai: 90% chance of rain tomorrow.
================================== Ai Message ==================================

[]
Tool Calls:
  move_meeting_online (call_445629230034958626)
 Call ID: call_445629230034958626
  Args:
    meeting: 10am standup
================================= Tool Message =================================
Name: move_meeting_online

Done. '10am standup' is now an online meeting.
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Yes, it is expected to rain in 